# DEMAND PATTERNS

# Imports

In [ ]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as f
import findspark
import plotly.express as px
import pandas as pd
import os
from pathlib import Path
import geopandas as gpd
from shapely import wkt
import json

# Paths

In [ ]:
projectRoot = Path.cwd().parents[1]

# Yellow

YELLOW_DATA_PATH = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "yellow"
YELLOW_TABLES_DIR = projectRoot / "data" / "plotTables" / "demandPatterns" / "yellow"
YELLOW_HTML_DIR = projectRoot / "data" / "demandPatterns" / "html" / "yellow"
YELLOW_JSON_DIR = projectRoot / "data" / "demandPatterns" / "json" / "yellow"

os.makedirs(YELLOW_TABLES_DIR, exist_ok=True)
os.makedirs(YELLOW_HTML_DIR, exist_ok=True)
os.makedirs(YELLOW_JSON_DIR, exist_ok=True)

# FHVHV

FHVHV_DATA_PATH = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
FHVHV_TABLES_DIR = projectRoot / "data" / "plotTables" / "demandPatterns" / "fhvhv"
FHVHV_HTML_DIR = projectRoot / "data" / "demandPatterns" / "html" / "fhvhv"
FHVHV_JSON_DIR = projectRoot / "data" / "demandPatterns" / "json" / "fhvhv"

os.makedirs(FHVHV_TABLES_DIR, exist_ok=True)
os.makedirs(FHVHV_HTML_DIR, exist_ok=True)
os.makedirs(FHVHV_JSON_DIR, exist_ok=True)

# Geo data

GEO_DATA = projectRoot / "data" / "NYC_Taxi_Zones_20251203.csv"


# Utils

In [ ]:
def save_table(df:pd.DataFrame, name:str, path: Path):
    path_table = os.path.join(str(path), f"{name}.csv")
    df.to_csv(path_table, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

def make_date_col(df):
    """
    Build a pandas datetime column from pickup_year + pickup_month.

    2020 01 -> 2020-01-01
    """
    df["date"] = pd.to_datetime(
        df["pickup_year"].astype(str) + "-" +
        df["pickup_month"].astype(str) + "-01"
    )
    return df

def add_temporal_columns(df: DataFrame) -> DataFrame:
    """
    Derive useful temporal features.
    """
    df = (
        df
        .withColumn("pickup_hour", f.hour("pickup_datetime"))
        .withColumn("pickup_day_of_week", f.dayofweek("pickup_datetime"))
        .withColumn("pickup_day_name", f.date_format("pickup_datetime", "EEEE"))
        .withColumn("pickup_is_weekend", f.dayofweek("pickup_datetime").isin([1, 7]))
        .withColumn("dropoff_year", f.year("dropoff_datetime"))
        .withColumn("dropoff_month", f.month("dropoff_datetime"))
        .withColumn("dropoff_hour", f.hour("dropoff_datetime"))
        .withColumn("dropoffp_day_of_week", f.dayofweek("dropoff_datetime"))
        .withColumn("dropoff_day_name", f.date_format("dropoff_datetime", "EEEE"))
        .withColumn("dropoff_is_weekend", f.dayofweek("dropoff_datetime").isin([1, 7]))
        .withColumn("time_of_day",
            f.when((f.col("pickup_hour") >= 6)  & (f.col("pickup_hour") < 10), "morning_rush")
             .when((f.col("pickup_hour") >= 10) & (f.col("pickup_hour") < 16), "midday")
             .when((f.col("pickup_hour") >= 16) & (f.col("pickup_hour") < 20), "evening_rush")
             .when((f.col("pickup_hour") >= 20) & (f.col("pickup_hour") < 24), "night")
             .otherwise("late_night")
        )
    )
    return df

In [ ]:
LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

TIME_ORDER = [
    "morning_rush", 
    "midday",   
    "evening_rush",       
    "night", 
    "late_night"          
]

COLOR_DICT = {
    "Very Low": "#E8F4F8",
    "Low": "#B3D9F2", 
    "Medium": "#F9E79F",
    "High": "#F5B041",      
    "Very High": "#E74C3C",   
    "Extremely High": "#943126",    
    "Saturated": "#641E16"     
}
ALL_LEVELS = ["Very Low", "Low", "Medium", "High", "Very High", "Extremely High", "Saturated"]

# Functions

In [ ]:
def percentiles(df, column):
    result = df.select(
        f.percentile_approx(column, [0.25, 0.50, 0.75, 0.85, 0.95, 0.99, 0.999, 0.9999])
        .alias("percentiles")
    ).collect()[0][0]
    
    percentiles_dict = {
        "p25": result[0],
        "p50": result[1],
        "p75": result[2],
        "p85": result[3],
        "p95": result[4],
        "p99": result[5],
        "p999": result[6],
        "p9999": result[7]
    }
    
    return percentiles_dict


In [ ]:
def add_demand_level(df: DataFrame, path: Path = None) -> tuple[DataFrame, dict, dict]:
    """Add demand level labels to trips based on percentile thresholds. Returns processed DataFrame and percentile dicts for trip counts and average demand."""

    df_grouped = (
        df
        .groupBy("time_of_day", "pickup_zone")
        .agg(
            f.count("*").alias("trip_number")
        )
        .orderBy(f.desc("trip_number"))
    )

    w = Window.partitionBy("time_of_day")

    df_grouped = (
        df_grouped
        .withColumn(
            "total_trips_per_time",
            f.sum("trip_number").over(w)
        )
        .withColumn(
            "avg_demand",
            f.round((f.col("trip_number")/f.col("total_trips_per_time"))*100, 4)
        )
    )

    p_dict = percentiles(df_grouped, "trip_number")
    avg_p_dict = percentiles(df_grouped, "avg_demand")

    df_res = (
        df_grouped
        .withColumn(
            "level",
            f.when(
                f.col("trip_number") <= p_dict['p25'], "Very Low"
            ).when(
                f.col("trip_number") <= p_dict['p50'], "Low"
            ).when(
                f.col("trip_number") <= p_dict['p75'], "Medium"
            ).when(
                f.col("trip_number") <= p_dict['p85'], "High"
            ).when(
                f.col("trip_number") <= p_dict['p95'], "Very High"
            ).when(
                f.col("trip_number") <= p_dict['p99'], "Extremely High"
            ).otherwise(
                "Saturated"
            )
        )
        .withColumn(
            "avg_level",
            f.when(
                f.col("avg_demand") <= avg_p_dict['p25'], "Very Low"
            ).when(
                f.col("avg_demand") <= avg_p_dict['p50'], "Low"
            ).when(
                f.col("avg_demand") <= avg_p_dict['p75'], "Medium"
            ).when(
                f.col("avg_demand") <= avg_p_dict['p85'], "High"
            ).when(
                f.col("avg_demand") <= avg_p_dict['p95'], "Very High"
            ).when(
                f.col("avg_demand") <= avg_p_dict['p99'], "Extremely High"
            ).otherwise(
                "Saturated"
            )
        )
        .toPandas()
    )

    save_table(df_res, f"demand_pattern_data", path)

    return df_res, p_dict, avg_p_dict
 

In [ ]:
def plot_demand_patterns(df: DataFrame, gdf: DataFrame, column:str, time_cat: str, title:str = "",save_paths: tuple[Path, Path] = None, table_path: Path = None):
    """Plot choropleth map showing demand patterns for a specific time category. Optionally save as HTML and JSON."""

    if table_path:
        df = load_csv(table_path)

    gdf_borough = gdf.dissolve(by="zone").reset_index()
    geojson = json.loads(gdf_borough.to_json())

    df = df[df["time_of_day"] == time_cat]



    fig = px.choropleth_map(
        df,
        geojson=geojson,
        locations="pickup_zone",              
        featureidkey="properties.zone",
        color=column,
        category_orders={column: ALL_LEVELS},
        color_discrete_map=COLOR_DICT,
        map_style="carto-positron",
        zoom=10,
        center={"lat": 40.7128, "lon": -74.0060},
        opacity=0.7,
        title=title,
    )

    categories_in_first_frame = set()
    for trace in fig.data:
        if hasattr(trace, 'name') and trace.name in ALL_LEVELS:
            categories_in_first_frame.add(trace.name)
            trace.showlegend = True
    for category in ALL_LEVELS:
        if category not in categories_in_first_frame:
            fig.add_trace({
                'type': 'choroplethmap',
                'locations': [None],
                'z': [None],
                'name': category,
                'legendgroup': category,
                'showlegend': True,
                'visible': True,
                'marker_line_width': 0,
                'coloraxis': None,
                'geojson': geojson,
                'featureidkey': "properties.zone",
                'colorscale': [[0, COLOR_DICT[category]], [1, COLOR_DICT[category]]],
                'showscale': False
            })

    fig.update_layout(
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        width=1200,
        height=800,
        legend=dict(
            title="Demand Level",
            traceorder="normal",
            itemsizing="constant"
        )
    )

    fig.show()
    if save_paths:
        fig.write_html(save_paths[0])
        fig.write_json(save_paths[1])

# Spark Init

## No need to run Spark if you run the code from *.csv

`NOTE`:
Change the configs according to your device if you wanna run it (if you don't have the plot tables already in your device).

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

# Load Geo Data

In [ ]:
df_temp = pd.read_csv(GEO_DATA)

df_temp = df_temp.rename(columns ={"Shape Geometry" : "geometry", "Shape Length" : "length", "Shape Area" : "area", "Location ID" : "id", "Zone": "zone", "Borough" : "borough"})
    # Convierte WKT → geometría
df_temp['geometry'] = df_temp['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(df_temp, crs='EPSG:4326')

gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x

# Yellow

## Load Data

In [ ]:
yellow = spark.read.parquet(str(YELLOW_DATA_PATH))
yellow = add_temporal_columns(yellow)

In [ ]:
yellow.count()

In [ ]:
yellow.printSchema()

In [ ]:
yellow_pd, _, _ = add_demand_level(yellow, YELLOW_TABLES_DIR)

## Plots

### Morning (6 A.M - 10 A.M)

In [ ]:
plot_demand_patterns(
    df=yellow_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="morning_rush",
    title="Yellow Demand Pattern (morning_rush)",
    save_paths=tuple((YELLOW_HTML_DIR / "yellow_demand_pattern_1_morning_rush.html", YELLOW_JSON_DIR / "yellow_demand_pattern_1_morning_rush.json"))
)

### Midday (10 A.M - 4 P.M)


In [ ]:
plot_demand_patterns(
    df=yellow_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="midday",
    title="Yellow Demand Pattern (midday)",
    save_paths=tuple((YELLOW_HTML_DIR / "yellow_demand_pattern_2_midday.html", YELLOW_JSON_DIR / "yellow_demand_pattern_2_midday.json"))
)

### Evening (4 P.M - 8 P.M)

In [ ]:
plot_demand_patterns(
    df=yellow_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="evening_rush",
    title="Yellow Demand Pattern (evening_rush)",
    save_paths=tuple((YELLOW_HTML_DIR / "yellow_demand_pattern_3_evening_rush.html", YELLOW_JSON_DIR / "yellow_demand_pattern_3_evening_rush.json"))
)

### Night (8 P.M - 12 A.M)

In [ ]:
plot_demand_patterns(
    df=yellow_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="night",
    title="Yellow Demand Pattern (night)",
    save_paths=tuple((YELLOW_HTML_DIR / "yellow_demand_pattern_4_night.html", YELLOW_JSON_DIR / "yellow_demand_pattern_4_night.json"))
)

### Late Night (12 A.M - 6 A.M)

In [ ]:
plot_demand_patterns(
    df=yellow_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="late_night",
    title="Yellow Demand Pattern (late_night)",
    save_paths=tuple((YELLOW_HTML_DIR / "yellow_demand_pattern_5_late_night.html", YELLOW_JSON_DIR / "yellow_demand_pattern_5_late_night.json"))
)

# FHVHV

## Load Data

In [ ]:
fhvhv = spark.read.parquet(str(FHVHV_DATA_PATH))
fhvhv = add_temporal_columns(fhvhv)

In [ ]:
fhvhv.count()

In [ ]:
fhvhv.printSchema()

In [ ]:
fhvhv_pd, _, _ = add_demand_level(fhvhv, FHVHV_TABLES_DIR)

## Plots

### Morning (6 A.M - 10 A.M)

In [ ]:
plot_demand_patterns(
    df=fhvhv_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="morning_rush",
    title="FHVHV Demand Pattern (morning_rush)",
    save_paths=tuple((FHVHV_HTML_DIR / "fhvhv_demand_pattern_1_morning_rush.html", FHVHV_JSON_DIR / "fhvhv_demand_pattern_1_morning_rush.json"))
)

### Midday (10 A.M - 4 P.M)


In [ ]:
plot_demand_patterns(
    df=fhvhv_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="midday",
    title="FHVHV Demand Pattern (midday)",
    save_paths=tuple((FHVHV_HTML_DIR / "fhvhv_demand_pattern_2_midday.html", FHVHV_JSON_DIR / "fhvhv_demand_pattern_2_midday.json"))
)

### Evening (4 P.M - 8 P.M)

In [ ]:
plot_demand_patterns(
    df=fhvhv_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="evening_rush",
    title="FHVHV Demand Pattern (evening_rush)",
    save_paths=tuple((FHVHV_HTML_DIR / "fhvhv_demand_pattern_3_evening_rush.html", FHVHV_JSON_DIR / "fhvhv_demand_pattern_3_evening_rush.json"))
)

### Night (8 P.M - 12 A.M)

In [ ]:
plot_demand_patterns(
    df=fhvhv_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="night",
    title="FHVHV Demand Pattern (night)",
    save_paths=tuple((FHVHV_HTML_DIR / "fhvhv_demand_pattern_4_night.html", FHVHV_JSON_DIR / "fhvhv_demand_pattern_4_night.json"))
)

### Late Night (12 A.M - 6 A.M)

In [ ]:
plot_demand_patterns(
    df=fhvhv_pd, 
    gdf=gdf, 
    column="level", 
    time_cat="late_night",
    title="FHVHV Demand Pattern (late_night)",
    save_paths=tuple((FHVHV_HTML_DIR / "fhvhv_demand_pattern_5_late_night.html", FHVHV_JSON_DIR / "fhvhv_demand_pattern_5_late_night.json"))
)

# Stop Spark

In [ ]:
spark.stop()